In [22]:
import numpy as np
import pyarrow.dataset as ds

def compute_feature_stats(
    paths,
    features,
    cat_cols,
    fold_col=None,
    include_folds=None,
    exclude_folds=None,
    batch_rows=200_000,
):
    num_cols = [c for c in features if c not in cat_cols]
    dataset = ds.dataset(paths, format="parquet")
    fexpr = None
    if fold_col:
        col = ds.field(fold_col)
        if include_folds is not None:
            fexpr = col.isin(sorted(include_folds))
        if exclude_folds is not None:
            ex = ~col.isin(sorted(exclude_folds))
            fexpr = ex if fexpr is None else (fexpr & ex)

    scanner = dataset.scanner(columns=num_cols, filter=fexpr, batch_size=batch_rows)

    n = 0
    s = None
    s2 = None
    for rb in scanner.to_reader():
        X = rb.to_pandas()[num_cols].to_numpy(dtype=np.float64, copy=False)  # float64で集計
        if s is None:
            d = X.shape[1]
            s = np.zeros(d, dtype=np.float64)
            s2 = np.zeros(d, dtype=np.float64)
        s += X.sum(axis=0)
        s2 += (X * X).sum(axis=0)
        n += X.shape[0]

    mean = s / max(n, 1)
    var = s2 / max(n, 1) - mean**2
    var[var < 0] = 0.0  # 数値誤差
    std = np.sqrt(var)
    std[std == 0] = 1.0  # 定数列保護
    return mean.astype(np.float32), std.astype(np.float32)


In [ ]:
from dataclasses import dataclass, field
import torch
import numpy as np

@dataclass(eq=False)
class ParquetStream(IterableDataset):
    ...
    # 追加：標準化オプション
    standardize: bool = False
    mean: np.ndarray | None = None
    std:  np.ndarray | None = None
    norm_indices: np.ndarray | None = None  # 標準化対象の列インデックス（数値列のみ）

    def __post_init__(self):
        super().__init__()
        ...
        # 標準化対象の列インデックス（cat_colsを除く）
        if self.standardize:
            cat_set = set(self.cat_cols or [])
            self._num_feats = [c for c in self.features if c not in cat_set]
            # self.features の中で、数値列の位置
            idx = [self.features.index(c) for c in self._num_feats]
            self.norm_indices = np.asarray(idx, dtype=np.int64)

            assert self.mean is not None and self.std is not None, \
                "standardize=True のときは mean/std を渡してください"
            # mean/std を features 順に並べ替える
            # ここでは self.mean/self.std は _num_feats 順で渡す前提
            self._mean = torch.tensor(self.mean, dtype=torch.float32)
            self._std  = torch.tensor(self.std,  dtype=torch.float32)

    def __iter__(self):
        ...
            for i0 in range(0, len(yb), self.batch_size):
                i1 = min(i0 + self.batch_size, len(yb))
                if self.rows_per_epoch and emitted >= self.rows_per_epoch:
                    return
                xb = torch.from_numpy(Xb[i0:i1]).float()  # (B, F)
                if self.standardize:
                    # 対象列だけ正規化（固定統計）
                    idx = self.norm_indices
                    xb[:, idx] = (xb[:, idx] - self._mean) / self._std

                ybt = torch.from_numpy(yb[i0:i1]).float()
                if wb is None:
                    yield xb, ybt
                else:
                    yield xb, ybt, torch.from_numpy(wb[i0:i1]).float()
                emitted += (i1 - i0)
        ...

In [42]:
rename_dict = {
    "cb_v1": "cb-001",
    "cb_v2": "cb-012",
    "lgbm_v1": "lgbm-001",
    "lgbm_v2": "lgbm-012",
    "lgbm_v3": "lgbm-023",
    "logreg_v1": "logreg-009",
    "logreg_v2": "logreg-017",
    "logreg_v3": "logreg-020",
    "mlp_v1": "mlp-002",
    "mlp_v2": "mlp-004",
    "mlp_v3": "mlp-005",
    "mlp_v4": "mlp-006",
    "mlp_v5": "mlp-007",
    "mlp_v6": "mlp-008",
    "mlp_v7": "mlp-010",
    "mlp_v8": "mlp-020",
    "mlp_v9": "mlp-024",
    "rfc_v1": "rfc-001",
    "rfc_v2": "rfc-012",
    "rfc_v3": "rfc-019",
    "rfc_v4": "rfc-021",
    "xgb_v1": "xgb-001",
    "xgb_v2": "xgb-003",
    "xgb_v3": "xgb-011",
    "xgb_v4": "xgb-012",
    "xgb_v5": "xgb-015",
    "xgb_v6": "xgb-013",
    "xgb_v7": "xgb-016",
    "xgb_v8": "xgb-019",
    "xgb_v9": "xgb-021",
}
# bulk_clone_studies(rename_dict, url)

In [43]:
for study in rename_dict.keys():
    optuna.delete_study(
        study_name=study, storage=url)

In [ ]:
import pyarrow.parquet as pq
import polars as pl

pf = pq.ParquetFile("data.parquet")
schema = pf.schema_arrow
for field in schema:
    if pa.types.is_dictionary(field.type):
        # dictionary のキー(=ユニーク値)は field.dictionary を読む
        dict_type = field.type
        print(field.name, dict_type.dictionary_length)  # ←ユニーク数

In [58]:
import pyarrow.parquet as pq
import polars as pl

data = pl.DataFrame(
    {
        "x": [1, 2, 3],
        "y": [4, 5, 6]
    }
)
data = data.with_columns(
    pl.col("x").cast(pl.Utf8).cast(pl.Categorical)
)
data.write_parquet("data.parquet")

In [59]:
p = pq.ParquetFile("data.parquet")

ArrowInvalid: Unrecognized type: 24

In [17]:
import pyarrow.parquet as pq
import polars as pl

data = pl.DataFrame(
    {
        "x": [1, 2, 3],
        "y": [4, 5, 6]
    }
)
data = data.with_columns(
    pl.col("x").cast(pl.Utf8).cast(pl.Categorical)
)
data.write_parquet("data2.parquet")

In [18]:
data = pl.read_parquet(["data.parquet", "data2.parquet"])

In [28]:
n_rows = (
    pl.scan_parquet(["data.parquet", "data2.parquet"])               # ← list[str]でもOK
      .select(pl.len())                             # 総行数を数える（列は不要）
      .collect(engine="streaming")                      # 大量ファイルでも軽量に
      .item()                                       # 1x1 -> Pythonのintへ
)

In [49]:
import numpy as np
y = pl.read_parquet("data2.parquet", columns=["y"]).get_column("y").cast(pl.Float32).to_numpy()

In [50]:
print(y, type(y))

[4. 5. 6.] <class 'numpy.ndarray'>


In [10]:
def cardinalities(path: str, cat_cols: list[str]) -> dict[str, int]:
    scan = pl.scan_parquet(path)
    return {c: scan.select(pl.col(c).n_unique()).collect().item() for c in cat_cols}

cardinalities("data2.parquet", "x")

{'x': 3}

In [8]:
data.schema

Schema([('x', Categorical), ('y', Int64)])

In [10]:
hdr = pl.read_parquet("data.parquet", n_rows=0)

In [12]:
def cardinalities(path: str, cat_cols: list[str]) -> dict[str, int]:
    scan = pl.scan_parquet(path)
    return {c: scan.select(pl.col(c).n_unique()).collect().item() for c in cat_cols}

In [14]:
cardinalities("data.parquet", "x")

{'x': 1}

In [18]:
from pathlib import Path
path = Path("abc/data.parquet")
print(path, path.resolve())

abc/data.parquet /home/hanse/kaggle/binary-bank/notebooks/abc/data.parquet


# ParquetIterator DEBUG

In [1]:
import os, gc, psutil
import pyarrow as pa
import pyarrow.dataset as ds

p = psutil.Process(os.getpid())


def rss():
    return round(p.memory_info().rss/1024**2, 1)


paths = ["../../artifacts/features/027/tr_df.parquet"]  # あなたのファイル
scanner = ds.dataset(paths, format="parquet").scanner(columns=["row_id"], batch_size=200_000)
r = scanner.to_reader()
print("A start  RSS(MB) =", rss(), "  Arrow(MB) =", pa.total_allocated_bytes()/1024**2)

for i in range(50):
    try:
        b = next(r)              # RecordBatch
    except StopIteration:
        break
    del b
    if i % 10 == 0:
        gc.collect()
        print(f"A step{i} RSS(MB) =", rss(), "  Arrow(MB) =", pa.total_allocated_bytes()/1024**2)

# 明示的クローズ＆掃除
if hasattr(r, "close"): r.close()
del r, scanner
gc.collect()
print("A end    RSS(MB) =", rss(), "  Arrow(MB) =", pa.total_allocated_bytes()/1024**2)

A start  RSS(MB) = 176.2   Arrow(MB) = 0.67352294921875
A step0 RSS(MB) = 196.8   Arrow(MB) = 4.78765869140625
A end    RSS(MB) = 196.8   Arrow(MB) = 0.0


In [1]:
# ==== B: cuDFネイティブのParquetチャンク読みで検証 ====
import os, gc, psutil
import cudf, cupy as cp, rmm
import polars as pl
import pyarrow.parquet as pq
from pathlib import Path

# ---- 設定 ----                  # Parquet のファイル/ディレクトリ
chunksize = "64MB"                # まずは 64MB 程度→ 32MB/16MB と調整可
TARGET = "target"                 # 必要なら
EXCLUDE = {"row_id"}              # 除外したい列
ROWGROUP_BATCH = 1
paths = ["../../artifacts/features/027/tr_df.parquet"]

# ---- RSSヘルパ ----
p = psutil.Process(os.getpid())
rss = lambda: round(p.memory_info().rss/1024**2, 1)

# ---- RMM 初期化（必ず一番最初に）----
rmm.reinitialize(pool_allocator=True, managed_memory=False)
try:
    from rmm.allocators.cupy import rmm_cupy_allocator   # 24/25系
    cp.cuda.set_allocator(rmm_cupy_allocator)
except Exception:
    print("WARN: rmm_cupy_allocator が見つからず、CuPy標準アロケータを使用します。")

# Pinnedメモリのプール（区切りで縮める用）
pmp = cp.cuda.PinnedMemoryPool()
cp.cuda.set_pinned_memory_allocator(pmp.malloc)

# ---- 特徴量リスト（ヘッダだけ Polars で読む）----
hdr = pl.read_parquet(paths, n_rows=0)
features = [c for c in hdr.columns if c not in EXCLUDE]

print("B start  RSS(MB) =", rss())

for path in map(str, paths):
    pf = pq.ParquetFile(path)
    nrg = pf.num_row_groups

    rg = 0
    while rg < nrg:
        # 1回で読む row group の束を決める
        bundle = list(range(rg, min(rg + ROWGROUP_BATCH, nrg)))
        rg = bundle[-1] + 1

        # cuDF で row group を直接読む（Arrow Table を経由しない）
        gdf = cudf.read_parquet(path, columns=features + [TARGET], row_groups=bundle)

        # --- ここで処理（例：形だけアクセス）---
        _ = int(gdf.shape[0])

        # 後始末（参照断つ→プール縮め→GC）
        del gdf
        pmp.free_all_blocks()
        cp.get_default_memory_pool().free_all_blocks()
        gc.collect()

        # ログ
        if rg % 5 == 0 or rg == nrg:
            print(f"[{Path(path).name}] rg<{rg}/{nrg}> RSS(MB) =", rss())

# 最終クリーンナップ
pmp.free_all_blocks()
cp.get_default_memory_pool().free_all_blocks()
gc.collect()
print("End   RSS(MB) =", rss())

B start  RSS(MB) = 518.1
[tr_df.parquet] rg<2/2> RSS(MB) = 705.1
End   RSS(MB) = 705.1


In [ ]:
cudf.read_parquet()

In [1]:
# ==== C: row group 単位で XGBoost 学習（即破棄）====
import os, gc, psutil
from pathlib import Path
import pyarrow.parquet as pq
import cudf, cupy as cp, rmm
import xgboost as xgb
import polars as pl

# --- 設定 ---
paths: list[str] = ["../../artifacts/features/027/tr_df.parquet"]
TARGET = "target"
EXCLUDE = {"row_id"}              # 除外列
ROWGROUP_BATCH = 1                # まずは1で安定確認（必要なら2以上に）
MAX_BIN = 256                     # 256→128/64で一時メモリを抑制

# --- ヘルパ ---
p = psutil.Process(os.getpid())
rss = lambda: round(p.memory_info().rss/1024**2, 1)
def gpumem_mb():
    free_b, total_b = cp.cuda.runtime.memGetInfo()
    return round((total_b - free_b)/1024**2, 1)

# --- RMM 初期化（必ず最初に）---
rmm.reinitialize(pool_allocator=True, managed_memory=False)
try:
    from rmm.allocators.cupy import rmm_cupy_allocator
    cp.cuda.set_allocator(rmm_cupy_allocator)
except Exception:
    print("WARN: rmm_cupy_allocator が見つからず、CuPy標準アロケータを使用します。")

pmp = cp.cuda.PinnedMemoryPool()
cp.cuda.set_pinned_memory_allocator(pmp.malloc)

# --- 特徴列（ヘッダだけ CPU で取得）---
hdr = pl.read_parquet(paths, n_rows=0)
# まずは数値列だけで安定化（カテゴリは後で enable_categorical=True で戻す）
numeric_cols = [c for c, t in hdr.schema.items()
                if c not in EXCLUDE and c != TARGET and t.is_numeric()]
assert len(numeric_cols) > 0, "数値の特徴列が0です"

print("C start  RSS(MB) =", rss(), " GPUused(MB) =", gpumem_mb(), " n_feats =", len(numeric_cols))

params = dict(
    tree_method="hist",
    device="cuda",
    objective="binary:logistic",   # 目的に合わせて
    max_depth=6,
    eta=0.3,
)

for path in map(str, paths):
    pf = pq.ParquetFile(path)
    nrg = pf.num_row_groups
    rg = 0
    while rg < nrg:
        bundle = list(range(rg, min(rg + ROWGROUP_BATCH, nrg)))
        rg = bundle[-1] + 1

        # cuDFで row group 直接読み（Arrow Table 経由なし）
        gdf = cudf.read_parquet(path, columns=numeric_cols + [TARGET], row_groups=bundle)

        # DMatrix を必要時だけ生成→すぐ学習→即破棄
        dtrain = xgb.QuantileDMatrix(
            data=gdf[numeric_cols],
            label=gdf[TARGET],
            max_bin=MAX_BIN,
            enable_categorical=False
        )
        bst = xgb.train(params, dtrain, num_boost_round=1)

        # 後始末（参照断つ→プール縮め→GC）
        del bst, dtrain, gdf
        pmp.free_all_blocks()
        cp.get_default_memory_pool().free_all_blocks()
        gc.collect()

        if rg % 5 == 0 or rg == nrg:
            print(f"[{Path(path).name}] rg<{rg}/{nrg}> RSS(MB) =", rss(), " GPUused(MB) =", gpumem_mb())

# 最終クリーンアップ
pmp.free_all_blocks()
cp.get_default_memory_pool().free_all_blocks()
gc.collect()
print("C end    RSS(MB) =", rss(), " GPUused(MB) =", gpumem_mb())

C start  RSS(MB) = 569.7  GPUused(MB) = 4656.5  n_feats = 2096
[tr_df.parquet] rg<2/2> RSS(MB) = 930.7  GPUused(MB) = 8187.5
C end    RSS(MB) = 930.7  GPUused(MB) = 8187.5


In [4]:
# ==== cuDF: row group 読み + プール上限 + ダウンキャスト ====
import os, gc, psutil
from pathlib import Path
import pyarrow.parquet as pq
import cudf, cupy as cp, rmm
import rmm.mr as mr
import polars as pl

# ---- 設定 ----
paths = ["../../artifacts/features/027/tr_df.parquet"]
TARGET = "target"
EXCLUDE = {"row_id"}
ROWGROUP_BATCH = 1
MAX_BIN = 64  # XGBで使うなら。ここでは読みテストなので未使用

# ---- RSS/GPUヘルパ ----
p = psutil.Process(os.getpid())
rss = lambda: round(p.memory_info().rss/1024**2, 1)
def gpumem_mb():
    free_b, total_b = cp.cuda.runtime.memGetInfo()
    return round((total_b - free_b)/1024**2, 1)

# ---- RMM 初期化（プールを“制限” or “無し”に）----
# A) まずはプール無し（縮みやすい）。速度犠牲あり
dev_mr = mr.CudaAsyncMemoryResource()
rmm.reinitialize(
    managed_memory=False,
    initial_pool_size=None,       # プールを使うなら "2GB" などに設定
)

# B) CuPy のプールも上限を設定（張り付き抑制）
try:
    from rmm.allocators.cupy import rmm_cupy_allocator
    cp.cuda.set_allocator(rmm_cupy_allocator)
except Exception:
    pass
# CuPy default pool の上限（例: 4GB）
cp.get_default_memory_pool().set_limit(4 * 1024**3)
pmp = cp.cuda.PinnedMemoryPool()
cp.cuda.set_pinned_memory_allocator(pmp.malloc)

# ---- 特徴量：数値だけ + ダウンキャスト前提 ----
hdr = pl.read_parquet(paths, n_rows=0)
# 数値列のみ（カテゴリや文字列は一旦外す：VRAMに効く）
numeric_cols = [c for c, t in hdr.schema.items()
                if c not in EXCLUDE and c != TARGET and t.is_numeric()]
if TARGET in hdr.columns:
    cols = numeric_cols + [TARGET]
else:
    cols = numeric_cols

print("Start  RSS(MB)=", rss(), " GPUused(MB)=", gpumem_mb(), " n_feats=", len(numeric_cols))

for path in map(str, paths):
    pf = pq.ParquetFile(path)
    nrg = pf.num_row_groups
    rg = 0
    while rg < nrg:
        bundle = list(range(rg, min(rg + ROWGROUP_BATCH, nrg)))
        rg = bundle[-1] + 1

        # row group 直読み（Arrow Table を経由しない）
        gdf = cudf.read_parquet(path, columns=cols, row_groups=bundle)

        # --- ダウンキャストで倍数削減（float64->float32, {u}int64->int32）---
        downcast = {}
        for c in numeric_cols:
            t = gdf[c].dtype
            if t.kind == "f" and t.itemsize > 4:
                downcast[c] = "float32"
            elif t.kind in ("i","u") and t.itemsize > 4:
                # 32bit に収まる範囲だけ downcast
                mn, mx = int(gdf[c].min()), int(gdf[c].max())
                if -2**31 <= mn and mx < 2**31:
                    downcast[c] = "int32"
        if downcast:
            gdf = gdf.astype(downcast)

        # ここで実処理…
        _ = int(gdf.shape[0])

        # 後始末：参照断つ→プール縮小→GC
        del gdf
        pmp.free_all_blocks()
        cp.get_default_memory_pool().free_all_blocks()
        gc.collect()

        print(f"[{Path(path).name}] rg<{rg}/{nrg}> RSS(MB)={rss()} GPUused(MB)={gpumem_mb()}")

# 最終クリーンアップ
pmp.free_all_blocks()
cp.get_default_memory_pool().free_all_blocks()
gc.collect()
print("End    RSS(MB)=", rss(), " GPUused(MB)=", gpumem_mb())

Start  RSS(MB)= 542.9  GPUused(MB)= 3330.5  n_feats= 2096
[tr_df.parquet] rg<1/2> RSS(MB)=732.2 GPUused(MB)=3346.5
[tr_df.parquet] rg<2/2> RSS(MB)=733.0 GPUused(MB)=3346.5
End    RSS(MB)= 733.0  GPUused(MB)= 3346.5


In [7]:
import rmm
import rmm.mr as mr
import cupy as cp

print("RMM ver:", rmm.__version__)
dev_mr = mr.CudaAsyncMemoryResource()
rmm.reinitialize(
    managed_memory=False,
    initial_pool_size=None,   # プール無し（戻りやすい）
)
mr.set_current_device_resource(dev_mr)

RMM ver: 25.08.00


In [9]:
paths = ["../../artifacts/features/027/tr_df.parquet"]
pf0 = pq.ParquetFile(paths[0])
all_cols = pf0.schema_arrow.names

print(len(all_cols))

2098


In [1]:
import subprocess
import json

def _gpu_free_by_pynvml():
    try:
        import pynvml as nvml
        nvml.nvmlInit()
        n = nvml.nvmlDeviceGetCount()
        frees = []
        for i in range(n):
            h = nvml.nvmlDeviceGetHandleByIndex(i)
            mem = nvml.nvmlDeviceGetMemoryInfo(h)  # bytes
            frees.append(mem.free / (1024 ** 3))   # -> GiB
        nvml.nvmlShutdown()
        return frees
    except Exception:
        return None

In [2]:
_gpu_free_by_pynvml()

[7.136760711669922]